## Reinforcement Finetuning of Retail Agent

Finetune a Retail Agent with Reinforcement learning


### Data

In [32]:
import json

with open("data/reinforcement_retail_agent/retail_agent_policy.md") as f:
    policy = f.read()

system_prompt_no_policy = "You are a helpful Retail Support Agent that helps customers with their orders, exchanges, returns and refunds."
system_prompt_with_policy = system_prompt_no_policy + "\n\n" + policy

with open("data/reinforcement_retail_agent/retail_agent_tools.json") as f:
    tool_definitions = json.load(f)

#### Prepare Evaluation data

In [33]:
EVAL_PARTIAL_FILE = "data/reinforcement_retail_agent/retail_agent_partial_eval.jsonl"
EVAL_NO_POLICY_FILE = "data/reinforcement_retail_agent/retail_agent_data_eval_no_policy.jsonl"
EVAL_WITH_POLICY_FILE = "data/reinforcement_retail_agent/retail_agent_data_eval_with_policy.jsonl"

with open(EVAL_PARTIAL_FILE) as eval_partial_file, \
        open(EVAL_NO_POLICY_FILE, "w") as eval_no_policy_file, \
        open(EVAL_WITH_POLICY_FILE, "w") as eval_with_policy_file:
    for line in eval_partial_file:
        item = json.loads(line)

        # Stringify tool calls and tool outputs, and fit them in assistant message for OpenAI Evaluation
        # which doesn't support structured tool calls and tool outputs
        input_items = []
        for msg in item["messages"]:
            if msg["role"] == "assistant" and msg.get("tool_calls"):
                for tc in msg["tool_calls"]:
                    fn = tc["function"]
                    call_id = tc.get("id", "")
                    input_items.append({
                        "role": "assistant",
                        "content": f"[tool_call:{call_id}] {fn['name']}({fn['arguments']})",
                    })
            elif msg["role"] == "tool":
                call_id = msg.get("tool_call_id", "")
                input_items.append({
                    "role": "assistant",
                    "content": f"[tool_result:{call_id}] {msg.get('content', '')}",
                })
            else:
                input_items.append(msg)
        item.pop("messages")
        item["input_items"] = input_items

        # Populate tool definitions
        item["tools"] = tool_definitions

        # Add reference response if not present
        item["reference_response"] = item.get("reference_response", "")

        # Write to eval files with system prompt
        item["input_items"][0]["content"] = system_prompt_no_policy
        eval_no_policy_file.write(json.dumps(item) + "\n")
        item["input_items"][0]["content"] = system_prompt_with_policy
        eval_with_policy_file.write(json.dumps(item) + "\n")

#### Prepare Training and Validation data

In [34]:
TRAINING_PARTIAL_FILE = "data/reinforcement_retail_agent/retail_agent_partial_train.jsonl"
TRAINING_FILE = "data/reinforcement_retail_agent/retail_agent_data_train.jsonl"
VALIDATION_FILE = "data/reinforcement_retail_agent/retail_agent_data_valid.jsonl"

def create_data_from_partial_file(partial_file_path, data_file_path):
    with open(partial_file_path) as partial_file, \
            open(data_file_path, "w") as data_file:
        for line in partial_file:
            item = json.loads(line)

            # Populate tool definitions
            item["tools"] = tool_definitions

            # Add reference response if not present
            item["reference_response"] = item.get("reference_response", "")

            # Write to training file with system prompt with policy
            item["messages"][0]["content"] = system_prompt_with_policy
            data_file.write(json.dumps(item) + "\n")


create_data_from_partial_file(TRAINING_PARTIAL_FILE, TRAINING_FILE)
create_data_from_partial_file(EVAL_PARTIAL_FILE, VALIDATION_FILE)  # reuse eval samples for validation


### Grader

In [12]:
import json
import os
import requests
from dotenv import load_dotenv

load_dotenv()

with open("tool_call_grader/tool_call_grader.py", "r") as f:
    grader_source = f.read()

grader = {
    "type": "python",
    "name": "tool_call_grader",
    "source": grader_source,
    "pass_threshold": 0.7,
}

headers = {
    "Authorization": f"Bearer {os.environ['AZURE_OPENAI_API_KEY']}",
}

validate_response = requests.post(os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/validate",
    headers=headers, json={ "grader": grader })

print(f"Status: {validate_response.status_code}")
print(json.dumps(validate_response.json(), indent=2))

Status: 200
{
  "grader": {
    "type": "python",
    "name": "tool_call_grader",
    "source": "import json\nfrom collections import Counter\nfrom typing import Any, Dict, List\n\nfrom rouge_score import rouge_scorer\n\n\nclass GraderConfig:\n    include_tools: List[str] = []\n    \"\"\"include_tools can be specified to grade only certain tools (by function name). Otherwise grader will grade all calls.\"\"\"\n\n\ndef grade(sample: Dict, item: Dict) -> float:\n    config = GraderConfig()\n    return grade_with_config(sample, item, config)\n\n\ndef grade_with_config(sample: Dict, item: Dict, config: GraderConfig) -> float:\n    actual_calls = _extract_actual_calls(sample)\n    expected_calls = item[\"reference_tool_calls\"]\n\n    if config.include_tools:\n        actual_calls = [c for c in actual_calls if c[\"function\"][\"name\"] in config.include_tools]\n        expected_calls = [c for c in item[\"reference_tool_calls\"] if c[\"function\"][\"name\"] in config.include_tools]\n\n    # 

In [24]:

run_payload = {
    "grader": grader,
    "model_sample": json.dumps({
        "output_tools": [
            { "id": "call_1", "type": "function", "function": { "name": "find_user_id_by_email", "arguments": json.dumps({"email": "user@example.com"}) } }
        ],
    }),
    "item": {
        "reference_tool_calls": [
            { "id": "call_1", "type": "function", "function": { "name": "find_user_id_by_email", "arguments": json.dumps({"email": "anotheruser@example.com"}) } }
        ],
    },
}

run_response = requests.post(os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/run",
    headers=headers, json=run_payload)

print(f"Status: {run_response.status_code}")
print(run_response.json())

Status: 200
{'reward': 0.55, 'metadata': {'name': 'tool_call_grader', 'type': 'python', 'errors': {'formula_parse_error': False, 'sample_parse_error': False, 'sample_parse_error_details': None, 'truncated_observation_error': False, 'unresponsive_reward_error': False, 'invalid_variable_error': False, 'invalid_variable_error_details': None, 'other_error': False, 'python_grader_server_error': False, 'python_grader_server_error_type': None, 'python_grader_runtime_error': False, 'python_grader_runtime_error_details': None, 'model_grader_server_error': False, 'model_grader_refusal_error': False, 'model_grader_refusal_error_details': None, 'model_grader_parse_error': False, 'model_grader_parse_error_details': None, 'model_grader_exceeded_max_tokens_error': False, 'model_grader_server_error_details': None, 'endpoint_grader_internal_error': False, 'endpoint_grader_internal_error_details': None, 'endpoint_grader_server_error': False, 'endpoint_grader_server_error_details': None, 'endpoint_grader

### Reinforcement FineTuning Job

In [55]:
import logging
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

logging.getLogger("httpx").setLevel(logging.WARNING)

client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/openai/v1/",
)

In [54]:
with open(TRAINING_FILE, "rb") as f:
    training_file = client.files.create(file=f, purpose="fine-tune")
with open(VALIDATION_FILE, "rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")

print(training_file.id, validation_file.id)

file-c943be5e9627414da84633482bcd83b1 file-f772e5eb7ef743cda57b01900e0b1f9d


In [27]:
from mcp_server.mcp_server import tool_endpoints

tools = tool_endpoints()  # be careful to not print tools as it contains mcp api key

In [38]:
job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id, 
    model="o4-mini",
    suffix="retail-agent",
    method=dict(
        type="reinforcement",
        reinforcement=dict(
            grader=grader,
            response_format=None,
            tools=tools,
            hyperparameters=dict(
                n_epochs=1,
                reasoning_effort="low",

                # eval_interval is number of training steps between evaluation runs.
                # Lower value means more frequent evaluations and more visibility into the learning process, but also longer training time.
                eval_interval=3,

                # eval_samples is number of evaluation samples to generate for every sample in validation file.
                # Every evaluation run will have # of samples = (# of validation file samples * eval_samples)
                # Higher value makes the validation curves more robust especially if there's stochasticity in model's outputs.
                # Evaluation score is averaged across multiple samples to reduce noise and reveal consistent patterns of learning.
                eval_samples=2,

                # compute_multiplier determines how hard should the model explore search space.
                # Higher value means diverse exploration (more rollouts).
                compute_multiplier=1.5,
            )
        )
    )
)

print(job.id)

HTTP Request: POST https://omi-ignite-demo-resource.openai.azure.com//openai/v1/fine_tuning/jobs "HTTP/1.1 201 Created"


ftjob-e7927c1ab119427f8fc8889d41845e7d


In [39]:
import time
from datetime import datetime
from IPython.display import display, clear_output

try:
    while job.status not in ["succeeded", "failed"]:
        events = client.fine_tuning.jobs.list_events(job.id)
        clear_output(wait=True)
        print("Latest job events:")
        for event in events.data[-3:]:
            local_time = datetime.fromtimestamp(event.created_at).strftime("%Y-%m-%d %H:%M:%S")
            print("-", local_time, event.message)
        time.sleep(10)

        job = client.fine_tuning.jobs.retrieve(job.id)
except KeyboardInterrupt:
    pass

priint("Job status:", job.status)

Latest job events:
- 2026-04-19 13:21:28 Preprocessing completed for file training file.
- 2026-04-19 13:19:27 Preprocessing running for file training file.
- 2026-04-19 13:19:00 Job enqueued. Waiting for jobs ahead to complete.


NameError: name 'priint' is not defined

### Evaluation

- https://developers.openai.com/cookbook/examples/evaluation/use-cases/mcp_eval_notebook
- https://developers.openai.com/cookbook/examples/evaluation/use-cases/completion-monitoring

In [ ]:
retail_agent_eval = client.evals.create(
    name="Retail Agent Evaluation",
    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "input_items": {"type": "array"},
                "reference_tool_calls": {"type": "array"},
                "reference_response": {"type": "string"},
            },
        },
        # include_sample_schema=True auto-resolves input messages via "item.input_items".
        # The field must be named "input_items" in item_schema and data, or run fails with: Unknown variable 'item.input_items'.
        "include_sample_schema": True,
    },
    testing_criteria=[grader],
)

eval_id = retail_agent_eval.id
print(eval_id)

HTTP Request: POST https://omi-ignite-demo-resource.openai.azure.com//openai/v1/evals "HTTP/1.1 201 Created"


eval_69e488a71ca08191bf22d8b3d137cdbd


In [37]:
mcp_server_url = os.environ["MCP_ENDPOINT"].rstrip("/") + "/mcp"
mcp_tool = {
    "type": "mcp",
    "server_label": "retail-mcp",
    "server_url": mcp_server_url,
    "headers": {
        "X-MCP-API-Key": os.environ["MCP_API_KEY"],
        "X-MCP-Read-Only": "true",
    },
    "require_approval": "never",
}

def create_run(name, items):
    return client.evals.runs.create(
        name=name,
        eval_id=eval_id,
        data_source={
            "type": "responses",
            "source": {
                "type": "file_content",
                "content": [{"item": item} for item in items],
            },
            "input_messages": {
                "type": "item_reference",
                "item_reference": "item.input_items",
            },
            "model": "o4-mini",
            "sampling_params": {
                "max_completions_tokens": 10000,
                "reasoning_effort": "low",
                "tools": [mcp_tool],
            },
        },
    )


with open(EVAL_NO_POLICY_FILE) as f:
    eval_no_policy_data = [json.loads(line) for line in f]
with open(EVAL_WITH_POLICY_FILE) as f:
    eval_with_policy_data = [json.loads(line) for line in f]

run_no_policy = create_run("o4-mini (no policy)", eval_no_policy_data)
run_with_policy = create_run("o4-mini (with policy)", eval_with_policy_data)

print(run_no_policy.id, run_with_policy.id)

HTTP Request: POST https://omi-ignite-demo-resource.openai.azure.com//openai/v1/evals/eval_69e488a71ca08191bf22d8b3d137cdbd/runs "HTTP/1.1 201 Created"
HTTP Request: POST https://omi-ignite-demo-resource.openai.azure.com//openai/v1/evals/eval_69e488a71ca08191bf22d8b3d137cdbd/runs "HTTP/1.1 201 Created"


evalrun_69e488ab8e1c8191b9d9b18c2b82fd6d evalrun_69e488adadf081918be7af51ed71bbcc


In [50]:
import time
from IPython.display import display, clear_output

def poll_runs(eval_id, run_ids):
    while True:
        runs = [client.evals.runs.retrieve(rid, eval_id=eval_id) for rid in run_ids]
        clear_output(wait=True)
        for run in runs:
            print(run.name, run.status, run.result_counts)
        if all(run.status in {"completed", "failed"} for run in runs):
            break
        time.sleep(20)

eval_id = retail_agent_eval.id
poll_runs(eval_id, [run_no_policy.id, run_with_policy.id])

o4-mini (no policy) completed ResultCounts(errored=0, failed=4, passed=4, total=8)
o4-mini (with policy) completed ResultCounts(errored=0, failed=1, passed=7, total=8)
